# LSSTCam FWHM vs Tower DIMM vs Portable DIMM

## Setup Notebook

In [ ]:
day_obs = 20251104

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np

from astropy.time import Time, TimeDelta
from datetime import datetime, timedelta, date

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from bokeh.io import output_notebook
from bokeh.plotting import figure, show
from bokeh.layouts import column, row
from bokeh.models import Span, Arrow, OpenHead, Label

from lsst_efd_client import EfdClient
from lsst.summit.utils import ConsDbClient, getAirmassSeeingCorrection, getBandpassSeeingCorrection
from lsst.summit.utils.dateTime import getDayObsStartTime, getDayObsEndTime, calcNextDay
from lsst.summit.utils.efdUtils import getEfdData

from scipy.stats import gaussian_kde

from tqdm.notebook import tqdm


# Update default colors to increase contrast
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=['#0072B2', '#D55E00', '#009E73'])

# Make sure Bokeh initializes properly
output_notebook()

# Define constants
SIGMA_TO_FWHM = 2 * np.sqrt(2 * np.log(2))
PIXEL_SCALE_ARCSEC = 0.2  # arcsec / pixel (LSSTCam)

# Create an EFD Client
efd_client = EfdClient("usdf_efd")

# Required to use ConsDb, following documentation above
os.environ["no_proxy"] += ",.consdb"

# Initialize ConsDb
cdb_client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

# Define the time stamps
t_start = getDayObsStartTime(day_obs)
t_end = getDayObsEndTime(day_obs)
print(f"Running analysis from {t_start} to {t_end}")

## Query the data

### Tower DIMM and Portable DIMM

In [ ]:
# Query all the DIMM data - It will contain data from both 
#  Tower DIMM (sal index 1) and Portable DIMM (sal index 2)
df_dimm_measurement = getEfdData(
    efd_client, 
    "lsst.sal.DIMM.logevent_dimmMeasurement", 
    columns=["salIndex", "fwhm", "secz"],
    begin=t_start, 
    end=t_end
)

if isinstance(df_dimm_measurement, dict):
    raise ValueError(f"Could not find DIMM data for day_obs={day_obs}")

# Drop invalid rows
df_dimm_measurement.dropna(inplace=True)

# Filter out seeing (FWHM) measurements that are too high
df_dimm_measurement = df_dimm_measurement[df_dimm_measurement["fwhm"] < 5]

# Correct seeing measurements due to airmass (secz)
df_dimm_measurement["fwhm_z"] = df_dimm_measurement["fwhm"] * df_dimm_measurement["secz"] ** (-3/5)

### LSSTCam Median FWHM at Zenith and for 500 nm

In [ ]:
# Let's query data from ConsDB
cdb_query = f"""
    SELECT
        e.seq_num AS seq,
        e.day_obs,
        q.physical_rotator_angle,
        e.altitude,
        e.airmass,
        e.obs_start,
        e.obs_end,
        e.focus_z,
        e.observation_reason,
        e.physical_filter as band_p,
        e.band,
        q.psf_sigma_median,
        q.aos_fwhm
    FROM
        cdb_lsstcam.visit1_quicklook AS q,
        cdb_lsstcam.exposure AS e
    WHERE
        q.visit_id = e.exposure_id
        AND (e.img_type = 'science' or e.img_type = 'acq')
        AND e.day_obs = {int(day_obs)}
"""

df_cdb = cdb_client.query(cdb_query).to_pandas()

# Drop any data with the following criteria
df_cdb = df_cdb[df_cdb['airmass'] != 0]
df_cdb = df_cdb[df_cdb['band_p'] != 'none']
df_cdb = df_cdb.reset_index(drop=True)  # Reset index after filtering

# Robust datetime parsing for mixed ISO8601 strings
df_cdb["obs_start"] = pd.to_datetime(df_cdb["obs_start"], format="ISO8601", errors="coerce")
df_cdb["obs_end"] = pd.to_datetime(df_cdb["obs_end"], format="ISO8601", errors="coerce")


def convert_psf_sigma_to_fwhm(psf_sigma: pd.Series, airmass: pd.Series, band_p: pd.Series) -> pd.Series:
    """
    Convert PSF sigma to FWHM.

    Parameters
    ----------
    psf_sigma : pd.Series
        The PSF sigma values in pixels.
    airmass : pd.Series
        The airmass values.
    band_p : pd.Series
        The physical filter names.

    Returns
    -------
    pd.Series
        The corresponding FWHM values at zenith for 500 nm.
    """
    # Convert PSF sigma (pixels) -> FWHM (arcsec)
    # NOTE: psf_sigma is the median sigma from visit1_quicklook.
    psf_fwhm = psf_sigma * SIGMA_TO_FWHM * PIXEL_SCALE_ARCSEC

    # Apply bandpass and airmass corrections
    airmass_correction = airmass.apply(getAirmassSeeingCorrection)
    bandpass_correction = band_p.apply(getBandpassSeeingCorrection)

    fwhm_zenith_500nm = psf_fwhm * airmass_correction * bandpass_correction
    return fwhm_zenith_500nm


# In theory, this convertion should be already done in ConsDB. However,
#  this column has not bee populated to this date. So we need to update 
#  our table manually.
df_cdb["fwhm_zenith_500nm_median"] = convert_psf_sigma_to_fwhm(
    psf_sigma=df_cdb["psf_sigma_median"],
    airmass=df_cdb["airmass"],
    band_p=df_cdb["band_p"],
)

### Wind Speed and Direction from Weather Tower

In [ ]:
t_night = Time(df_cdb["obs_start"], scale="utc")
t_night = t_night[t_night > t_start]
t_night = t_night[t_night < t_end]

if t_night.size == 0:
    raise ValueError(f"Missing data for day_obs={day_obs}")

t_night_begin = t_night.min().isot
t_night_end = t_night.max().isot

wind_query = f"""
    SELECT 
        mean("direction") AS "mean_direction",
        mean("speed") AS "mean_speed"
    FROM 
        "efd"."autogen"."lsst.sal.ESS.airFlow" 
    WHERE 
        time > '{t_night_begin}Z' 
        AND time < '{t_night_end}Z' 
        AND salIndex = 301 
    GROUP BY time(30s) 
    FILL(null)
"""

df_ess_airflow = await efd_client.influx_client.query(wind_query)

if isinstance(df_ess_airflow, dict):
    raise ValueError(f"No wind data for dayobs={day_obs}")

## Plot the data

### Timeline showing different seeing measurements

In [ ]:
# Split dimm data into two
df_tower_dimm = df_dimm_measurement[df_dimm_measurement["salIndex"] == 1]
df_portable_dimm = df_dimm_measurement[df_dimm_measurement["salIndex"] == 2]

# We can downsample the data since each exposure needs > 30 
df_tower_5s = df_tower_dimm.resample('5s').mean().dropna()
df_portable_5s = df_portable_dimm.resample('5s').mean().dropna()

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(1, 1, dpi=128, figsize=(14, 6))

ax.scatter(
    df_tower_dimm.index,
    df_tower_dimm.fwhm_z,
    s=3,
    color="C0",
    label="Tower DIMM (sal index = 1)",
)

ax.scatter(
    df_portable_dimm.index,
    df_portable_dimm.fwhm_z,
    s=3,
    color="C1",
    label="Portable DIMM (sal index = 2)",
)

ax.scatter(
    df_cdb.obs_end,
    df_cdb.fwhm_zenith_500nm_median,
    s=3,
    color="C2",
    label="Median FWHM from LSSTCam",
    alpha=0.5
)

ax.set_xlabel("Time")
ax.set_ylabel("z-corrected FWHM (arcsec)")

plt.legend()
plt.show()

### Correlation Plots - Raw

In [ ]:
# Ensure all are sorted by index
df_base = df_cdb[['obs_start', 'fwhm_zenith_500nm_median']].copy()
df_base['obs_start'] = pd.to_datetime(df_base['obs_start'])
df_base = df_base.set_index('obs_start').sort_index()
df_base.index = df_base.index.tz_localize('UTC')

df_t = df_tower_dimm[['fwhm_z']].rename(columns={'fwhm_z': 'tower_fwhm_z'}).sort_index()
df_p = df_portable_dimm[['fwhm_z']].rename(columns={'fwhm_z': 'portable_fwhm_z'}).sort_index()
df_w = df_ess_airflow[['mean_speed', 'mean_direction']].sort_index()

# Resample DIMM and wind to smooth out before matching
# NOTE: Resampling wind direction with .mean() is not strictly correct
#  for angular data (e.g., averaging 350° and 10° gives 180° instead of 0°).
#  Consider using circular statistics if precision matters.
df_t = df_t.resample('30s').mean().dropna()
df_p = df_p.resample('30s').mean().dropna()
df_w = df_w.resample('30s').mean().dropna()

# Merge all onto df_base with a tolerance window
tol = pd.Timedelta(seconds=30)

df_merged = pd.merge_asof(df_base, df_t, left_index=True, right_index=True, tolerance=tol, direction='nearest')
df_merged = pd.merge_asof(df_merged, df_p, left_index=True, right_index=True, tolerance=tol, direction='nearest')
df_merged = pd.merge_asof(df_merged, df_w, left_index=True, right_index=True, tolerance=tol, direction='nearest')

df_merged = df_merged.dropna()

In [ ]:
def triangle_plot(df, cols=None, figsize=(12, 12), s=3, alpha=0.25, color='steelblue', bins=30):
    if cols is None:
        cols = df.columns.tolist()
    data = df[cols].dropna()
    n = len(cols)
    fig, axes = plt.subplots(n, n, figsize=figsize)

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if j > i:
                ax.set_visible(False)
            elif i == j:
                ax.hist(data[cols[i]], bins=bins, alpha=alpha, color=color)
            else:
                ax.scatter(data[cols[j]], data[cols[i]], s=s, alpha=alpha, color=color)
                ax.grid(True, linestyle=':', alpha=0.3)
                r = data[cols[j]].corr(data[cols[i]])
                ax.annotate(f'r={r:.2f}', xy=(0.95, 0.95), xycoords='axes fraction',
                            ha='right', va='top', fontsize=9,
                            bbox=dict(boxstyle='round', fc='white', alpha=0.8))
                try:
                    x = data[cols[j]].astype(float).values
                    y = data[cols[i]].astype(float).values
                    xy = np.vstack([x, y])
                    kde = gaussian_kde(xy)
                    xi = np.linspace(x.min(), x.max(), 100)
                    yi = np.linspace(y.min(), y.max(), 100)
                    Xi, Yi = np.meshgrid(xi, yi)
                    Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
                    ax.contour(Xi, Yi, Zi, levels=5, colors='k', linewidths=0.5, alpha=0.7)
                except np.linalg.LinAlgError:
                    pass  # Skip if KDE fails (e.g., singular matrix)

            if i == n - 1:
                ax.set_xlabel(cols[j], fontsize=9)
            else:
                ax.set_xticklabels([])
            if j == 0:
                ax.set_ylabel(cols[i], fontsize=9)
            else:
                ax.set_yticklabels([])

    plt.tight_layout()
    return fig, axes

In [ ]:
fig, axes = triangle_plot(df_merged)
plt.show()

### Correlation Plots - Seeing Differences

In [ ]:
df_merged["tower_dimm_minus_portable_dimm_fwhm_z"] = df_merged.tower_fwhm_z - df_merged.portable_fwhm_z
df_merged["lsstcam_minus_tower_dimm_fwhm_z"] = df_merged.fwhm_zenith_500nm_median - df_merged.tower_fwhm_z
df_merged["lsstcam_minus_portable_dimm_fwhm_z"] = df_merged.fwhm_zenith_500nm_median - df_merged.portable_fwhm_z

In [ ]:
plot_cols = ['tower_dimm_minus_portable_dimm_fwhm_z', 'lsstcam_minus_tower_dimm_fwhm_z', 'lsstcam_minus_portable_dimm_fwhm_z', 'mean_speed']
df_plot = df_merged[plot_cols].dropna()

In [ ]:
fig, axes = triangle_plot(df_plot, color="firebrick")
plt.show()

### Timeline Plots with Differences

In [ ]:
def make_compass(size=120):
    
    p = figure(width=size, height=size, x_range=(-1.5, 1.5), y_range=(-1.5, 1.5),
           toolbar_location=None, min_border=0, match_aspect=True,
           sizing_mode='fixed')
    
    p.axis.visible = False
    p.grid.visible = False
    p.outline_line_color = None

    # Arrows: N(up), E(right), S(down), W(left)
    head_kw = dict(size=8, line_color='green')
    
    for dx, dy, label, tx, ty in [
        (0, 1, 'N', 0, 1.3),
        (1, 0, 'E', 1.3, 0),
        (0, -1, 'S', 0, -1.3),
        (-1, 0, 'W', -1.3, 0),
    ]:
        p.add_layout(Arrow(end=OpenHead(**head_kw), line_color='green',
                           x_start=0, y_start=0, x_end=dx, y_end=dy))
        p.add_layout(Label(x=tx, y=ty, text=label, text_align='center',
                           text_baseline='middle', text_font_size='10pt', text_color='green'))
    
    return p



def timeline_plot(df, diff_cols, wind_speed_col='mean_speed', wind_dir_col='mean_direction',
                  width=1000, height=300, s=3, colors=None):
    if colors is None:
        colors = ['#0072B2', '#D55E00', '#009E73']

    p1 = figure(width=width, height=height, x_axis_type='datetime',
                y_axis_label='ΔFWHM (arcsec)', tools='pan,xbox_zoom,xwheel_zoom,reset,save')

    for col, color in zip(diff_cols, colors):
        p1.scatter(df.index, df[col], size=s, color=color, alpha=0.7, legend_label=col)

    p1.add_layout(Span(location=0, dimension='width', line_color='black', line_dash='dashed', line_width=0.5))
    p1.legend.click_policy = 'hide'
    p1.legend.label_text_font_size = '8pt'

    p2 = figure(width=width, height=height, x_axis_type='datetime',
                x_range=p1.x_range,
                x_axis_label='Time', y_axis_label='Wind Speed (m/s)',
                tools='pan,xbox_zoom,xwheel_zoom,reset,save')

    speed = df[wind_speed_col].values
    direction = np.deg2rad(270.0 - df[wind_dir_col].values)
    scale = 0.3
    dx = np.cos(direction) * speed * scale
    dy = np.sin(direction) * speed * scale

    x0 = df.index
    y0 = speed
    dx_ms = dx * 60_000
    x1 = x0 + pd.to_timedelta(dx_ms, unit='ms')
    y1 = y0 + dy

    p2.segment(x0=x0, y0=y0, x1=x1, y1=y1, color='green', line_width=1, alpha=0.7)
    p2.scatter(x0, y0, size=1, color='green', alpha=0.5)

    compass = make_compass()
    layout = column(p1, row(p2, compass, sizing_mode='stretch_width'))
    show(layout)
    return p1, p2

In [ ]:
p1, p2 = timeline_plot(
    df_merged,
    diff_cols=['tower_dimm_minus_portable_dimm_fwhm_z', 'lsstcam_minus_tower_dimm_fwhm_z', 'lsstcam_minus_portable_dimm_fwhm_z'],
)

## Polar Plots

In [ ]:
def polar_seeing_plot(df, seeing_col, direction_col='mean_direction', speed_col='mean_speed',
                      figsize=(8, 8), s=20, alpha=0.5, vmin=None, vmax=None):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    theta = np.deg2rad(data[direction_col].values)
    r = data[seeing_col].values
    speed = data[speed_col].values

    if vmin is None:
        vmin = int(np.floor(speed.min()))
    if vmax is None:
        vmax = int(np.ceil(speed.max()))

    # Discrete colormap in 1 m/s steps
    bounds = np.arange(vmin, vmax + 1, 2)
    norm = mcolors.BoundaryNorm(bounds, ncolors=256)

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    full_viridis = plt.cm.viridis
    truncated_viridis = mcolors.LinearSegmentedColormap.from_list(
        'viridis_trunc', full_viridis(np.linspace(0.2, 1.0, 256))
    )

    sc = ax.scatter(theta, r, c=speed, cmap=truncated_viridis, norm=norm, s=s, alpha=alpha)

    cbar = plt.colorbar(sc, ax=ax, pad=0.1, ticks=bounds)
    cbar.set_label('Wind Speed (m/s)')

    ax.set_ylabel(seeing_col, labelpad=30)
    ax.set_title(f'Seeing vs Wind Direction\n({seeing_col})', pad=20)

    plt.tight_layout()
    return fig, ax

### Tower Dimm Seeing

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='tower_fwhm_z', alpha=1)

### Portable DIMM Seeing

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='portable_fwhm_z', alpha=1)

### LSSTCam Median FWHM at Zenith and 500 nm

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='fwhm_zenith_500nm_median', alpha=1)

### Tower DIMM - Portable DIMM

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='tower_dimm_minus_portable_dimm_fwhm_z', alpha=1)

### LSSTCam - Tower DIMM

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='lsstcam_minus_tower_dimm_fwhm_z', alpha=1)

### LSSTCam - Portable DIMM

In [ ]:
fig, ax = polar_seeing_plot(df_merged, seeing_col='lsstcam_minus_portable_dimm_fwhm_z', alpha=1)

## Binned heatmap (2D histogram in polar coordinates)

In [ ]:
def polar_binned_plot(df, seeing_col, direction_col='mean_direction', speed_col='mean_speed',
                      n_angle_bins=36, n_radial_bins=20, figsize=(8, 8), cmap='viridis'):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    angle_bins = np.linspace(0, 360, n_angle_bins + 1)
    radial_bins = np.linspace(data[seeing_col].min(), data[seeing_col].max(), n_radial_bins + 1)

    data['angle_bin'] = pd.cut(data[direction_col], bins=angle_bins, labels=False)
    data['radial_bin'] = pd.cut(data[seeing_col], bins=radial_bins, labels=False)

    grouped = data.groupby(['angle_bin', 'radial_bin'])[speed_col].mean().reset_index()

    theta = np.deg2rad((angle_bins[:-1] + angle_bins[1:]) / 2)
    r = (radial_bins[:-1] + radial_bins[1:]) / 2
    dtheta = np.deg2rad(360 / n_angle_bins)
    dr = radial_bins[1] - radial_bins[0]

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    for _, row in grouped.iterrows():
        ai, ri = int(row['angle_bin']), int(row['radial_bin'])
        ax.bar(theta[ai], dr, width=dtheta, bottom=r[ri] - dr/2,
               color=plt.cm.viridis(plt.Normalize(data[speed_col].min(), data[speed_col].max())(row[speed_col])),
               alpha=0.8, edgecolor='none')

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(data[speed_col].min(), data[speed_col].max()))
    cbar = plt.colorbar(sm, ax=ax, pad=0.1)
    cbar.set_label(f'Mean Wind Speed (m/s)')
    ax.set_title(f'{seeing_col}', pad=20)
    plt.tight_layout()
    return fig, ax

### Tower Dimm Seeing

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='tower_fwhm_z')

### Portable DIMM Seeing

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='portable_fwhm_z')

### LSSTCam Median FWHM at Zenith and 500 nm

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='lsstcam_minus_tower_dimm_fwhm_z')

### Tower DIMM - Portable DIMM

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='tower_dimm_minus_portable_dimm_fwhm_z')

### LSSTCam - Tower DIMM

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='lsstcam_minus_tower_dimm_fwhm_z')

### LSSTCam - Portable DIMM

In [ ]:
fig, ax = polar_binned_plot(df_merged, seeing_col='lsstcam_minus_portable_dimm_fwhm_z')

## Contour plot in polar coordinates


In [ ]:
def polar_contour_plot(df, seeing_col, direction_col='mean_direction', speed_col='mean_speed',
                       figsize=(8, 8), n_angle=72, n_radial=50):
    data = df[[seeing_col, direction_col, speed_col]].dropna()

    theta_grid = np.linspace(0, 2 * np.pi, n_angle)
    r_grid = np.linspace(data[seeing_col].min(), data[seeing_col].max(), n_radial)
    T, R = np.meshgrid(theta_grid, r_grid)

    from scipy.stats import binned_statistic_2d
    stat, _, _, _ = binned_statistic_2d(
        np.deg2rad(data[direction_col].values), data[seeing_col].values,
        data[speed_col].values, statistic='mean', bins=[theta_grid, r_grid]
    )

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    cs = ax.pcolormesh(T, R, stat.T, cmap='viridis', shading='auto')
    cbar = plt.colorbar(cs, ax=ax, pad=0.1)
    cbar.set_label('Mean Wind Speed (m/s)')
    ax.set_title(f'{seeing_col}', pad=20)
    plt.tight_layout()
    return fig, ax

In [ ]:
fig, ax = polar_contour_plot(df_merged, seeing_col='tower_fwhm_z')